In [0]:
import pandas as pd
import numpy as np

In [0]:
df = spark.table("workspace.default.features").toPandas()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228904 entries, 0 to 228903
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   air_store_id    228904 non-null  object        
 1   visit_date      228904 non-null  datetime64[ns]
 2   visitors        228904 non-null  int32         
 3   calendar_date   228904 non-null  datetime64[ns]
 4   day_of_week     228904 non-null  object        
 5   holiday_flg     228904 non-null  int32         
 6   air_genre_name  228904 non-null  object        
 7   air_area_name   228904 non-null  object        
 8   latitude        228904 non-null  float64       
 9   longitude       228904 non-null  float64       
 10  month           228904 non-null  int32         
 11  is_weekend      228904 non-null  int64         
 12  lag_1           228904 non-null  float64       
 13  lag_7           228904 non-null  float64       
 14  lag_14          228904 non-null  flo

In [0]:
df = df.drop(columns=["calendar_date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228904 entries, 0 to 228903
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   air_store_id    228904 non-null  object        
 1   visit_date      228904 non-null  datetime64[ns]
 2   visitors        228904 non-null  int32         
 3   day_of_week     228904 non-null  object        
 4   holiday_flg     228904 non-null  int32         
 5   air_genre_name  228904 non-null  object        
 6   air_area_name   228904 non-null  object        
 7   latitude        228904 non-null  float64       
 8   longitude       228904 non-null  float64       
 9   month           228904 non-null  int32         
 10  is_weekend      228904 non-null  int64         
 11  lag_1           228904 non-null  float64       
 12  lag_7           228904 non-null  float64       
 13  lag_14          228904 non-null  float64       
 14  lag_28          228904 non-null  flo

In [0]:
test_set_size = 35
last_date = df["visit_date"].max()
test_set_start = last_date - pd.Timedelta(days=test_set_size)

In [0]:
is_test = df["visit_date"] >= test_set_start
print("train set size: ", (~is_test).sum())
print("test set size: ", is_test.sum())

train set size:  203245
test set size:  25659


In [0]:
prev_week = df[["air_store_id", "visit_date", "visitors"]].copy()
prev_week.head()


,air_store_id,visit_date,visitors
0,air_dfe068a1bf85f395,2017-03-08,51
1,air_dfe068a1bf85f395,2017-03-09,40
2,air_dfe068a1bf85f395,2017-03-10,47
3,air_dfe068a1bf85f395,2017-03-11,32
4,air_dfe068a1bf85f395,2017-03-12,24


In [0]:
prev_week["visit_date"] = prev_week["visit_date"] + pd.Timedelta(days=7)
prev_week = prev_week.rename(columns={"visitors": "pred_seasonal_naive"})
prev_week.head()

,air_store_id,visit_date,pred_seasonal_naive
0,air_dfe068a1bf85f395,2017-03-15,51
1,air_dfe068a1bf85f395,2017-03-16,40
2,air_dfe068a1bf85f395,2017-03-17,47
3,air_dfe068a1bf85f395,2017-03-18,32
4,air_dfe068a1bf85f395,2017-03-19,24


In [0]:
df = df.merge(prev_week, on=["air_store_id", "visit_date"], how="left")
df.head()

,air_store_id,visit_date,visitors,day_of_week,holiday_flg,air_genre_name,air_area_name,latitude,longitude,month,is_weekend,lag_1,lag_7,lag_14,lag_28,roll_mean_7,roll_std_7,roll_mean_28,roll_std_28,y,pred_seasonal_naive
0,air_dfe068a1bf85f395,2017-03-08,51,Wednesday,0,Italian/French,Tōkyō-to Chūō-ku Ginza,35.672114,139.770825,3,0,42.0,44.0,32.0,36.0,34.000000,11.372481,34.250000,10.265458,3.951244,26.0
1,air_dfe068a1bf85f395,2017-03-09,40,Thursday,0,Italian/French,Tōkyō-to Chūō-ku Ginza,35.672114,139.770825,3,0,51.0,26.0,42.0,22.0,35.000000,12.635928,34.785714,10.740567,3.713572,22.0
2,air_dfe068a1bf85f395,2017-03-10,47,Friday,0,Italian/French,Tōkyō-to Chūō-ku Ginza,35.672114,139.770825,3,0,40.0,22.0,31.0,31.0,37.000000,12.069245,35.428571,10.482538,3.871201,50.0
3,air_dfe068a1bf85f395,2017-03-11,32,Saturday,0,Italian/French,Tōkyō-to Chūō-ku Ginza,35.672114,139.770825,3,1,47.0,50.0,34.0,30.0,40.571429,10.485818,36.000000,10.666667,3.496508,32.0
4,air_dfe068a1bf85f395,2017-03-12,24,Sunday,0,Italian/French,Tōkyō-to Chūō-ku Ginza,35.672114,139.770825,3,1,32.0,32.0,51.0,40.0,38.000000,9.983319,36.071429,10.631639,3.218876,22.0


In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [0]:
test = df[is_test & df["pred_seasonal_naive"].notna()]
y_true = test["visitors"].values 
y_pred = test["pred_seasonal_naive"].values 

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
rmsle = np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

print("Baseline (seasonal naive) on test set")
print(f"  MAE   = {mae:.3f}")
print(f"  RMSE  = {rmse:.3f}")
print(f"  RMSLE = {rmsle:.4f}")

Baseline (seasonal naive) on test set
  MAE   = 9.398
  RMSE  = 14.932
  RMSLE = 0.6683


## Baseline: seasonal naive (same weekday, last week)

**Scores to beat:**

| Metric | Value | What it means |
|--------|-------|---------------|
| **RMSLE** | **0.6683** | Primary metric |
| MAE | 9.398 | On average, off by ~9.4 visitors |
| RMSE | 14.932 | Matches observed right skew |